# SGLang Bridge — Integration Test

End-to-end validation of `boot_sglang` on a real GPU. Runs the v4 Driver protocol's SGLang backend against `meta-llama/Llama-3.2-1B`, compares per-hook activations and next-token argmax against the HF transformers backend, and exercises the affine intervention path.

Sibling notebook to `vLLM_Bridge_Integration_Test.ipynb`. Same acceptance gates, different inference engine (RadixAttention + Piecewise CUDA Graph instead of PagedAttention + classic CUDA graphs).

## Why both?

vLLM and SGLang specialize differently:
- **vLLM** — PagedAttention's per-token KV layout; best for general high-throughput inference.
- **SGLang** — RadixAttention's radix-tree prefix cache; best when many requests share long common prefixes (SAE training data with shared system prompts, agentic eval workloads).

Both extend past observation-only scope: each capture hook applies an affine transform (`output = output * scale + bias`, default identity) and returns the modified tensor, so interventions propagate to downstream layers.

## Acceptance gates
1. `boot_sglang` returns a `RemoteBridge` end-to-end.
2. Greedy next-token argmax matches `boot_transformers` exactly.
3. Per-hook L2 distance vs `boot_transformers` < `5e-3` (relative).
4. Affine intervention path: suppress / scale / add / set each shifts the next-token argmax and reverts.
5. Hook fires exactly once per forward (no Piecewise CUDA Graph double-fire).
6. `bridge.close()` releases GPU memory + tears down SGLang's scheduler subprocesses.

## Setup

Install SGLang 0.5.12.post1 (the version the plugin's monkey-patch + RPC walks were validated against). SGLang's hard `timm==1.0.16` pin conflicts with the TransformerLens `multimodal` extra, so we install SGLang into a clean env.

In [ ]:
# Install sglang and TransformerLens @ feature/SGLang-driver. ~3-5 minutes.
# Branch must match this notebook: feature/SGLang-driver has the SGLangDriver and
# the tl_bridge_sglang inspect provider. main / dev-4.x lack both.
# sglang pinned to 0.5.12.post1 — the version the internal-API walks in
# transformer_lens/model_bridge/sources/sglang/{plugin,internals}.py were validated against.
# SGLang rearranges its internal class paths every 1-3 weeks; re-validate before bumping.
%pip install -q "sglang==0.5.12.post1"
%pip install -q git+https://github.com/TransformerLensOrg/TransformerLens.git@feature/SGLang-driver

In [ ]:
import gc
import os
import sys

import torch

# HF_TOKEN comes from Colab secrets. Falls back to env var for non-Colab runs.
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except (ImportError, Exception):
    pass
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets (gear icon, left sidebar)."

# Read the installed sglang version from package metadata, NOT `import sglang` —
# importing sglang triggers CUDA C extension loading. Metadata read works regardless.
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as _pkg_version

_PINNED_SGLANG = "0.5.12.post1"
try:
    _sglang_ver = _pkg_version("sglang")
    print(f"sglang version: {_sglang_ver}")
    if _sglang_ver != _PINNED_SGLANG:
        print(
            f"⚠ expected sglang=={_PINNED_SGLANG} (the version the capture plugin is validated "
            f"against); got {_sglang_ver}. Newer versions may move SGLang internals the plugin walks. "
            f"Re-pin with %pip install -q 'sglang=={_PINNED_SGLANG}' and restart the runtime."
        )
except PackageNotFoundError:
    print("⚠ sglang is not installed — run the install cell above.")

MODEL = "meta-llama/Llama-3.2-1B"
PROMPT = "The quick brown fox jumps over the"
DTYPE = torch.float16
torch.manual_seed(0)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — abort'}")
assert torch.cuda.is_available(), "GPU runtime required."

## Step 1 — HF reference

Capture activations from the HF transformers backend (the boot_transformers driver) as the ground truth for Steps 4 & 5.

In [ ]:
from transformer_lens.model_bridge.transformer_bridge import TransformerBridge

hf_bridge = TransformerBridge.boot_transformers(MODEL, dtype=DTYPE, device="cuda")
tokens = hf_bridge.to_tokens(PROMPT)
print(f"Tokens: shape={tuple(tokens.shape)}, first={tokens[0, :5].tolist()}")

hf_logits, hf_cache = hf_bridge.run_with_cache(tokens)
argmax_hf = int(hf_logits[0, -1].argmax())
print(f"HF next-token argmax: {argmax_hf} → {hf_bridge.tokenizer.decode([argmax_hf])!r}")

hf_acts = {k: v.float().cpu() for k, v in hf_cache.items()}
del hf_bridge, hf_logits, hf_cache
gc.collect()
torch.cuda.empty_cache()

## Step 2 — Boot SGLang bridge

`boot_sglang` constructs an SGLang `Engine` and wraps it in a `RemoteBridge` via `SGLangDriver`. The plugin's monkey-patched `ModelRunner.load_model` installs capture hooks before Piecewise CUDA Graph capture, so hooks survive the compiled forward.

In [ ]:
from transformer_lens.model_bridge.remote_bridge import RemoteBridge

bridge = RemoteBridge.boot_sglang(
    MODEL,
    dtype=DTYPE,
    # Smaller KV cache than the model's native context — fits on a single L4 / A100 40 GB.
    max_total_tokens=2048,
)
print(f"Bridge: {type(bridge).__name__}")
print(f"Fireable hooks: {len(bridge._driver.supported_hook_points)}")
print(f"Non-fireable (SGLang gates): {sorted(h for h in bridge._driver.non_fireable_hook_points if h.startswith('blocks.0'))}")

## Step 3 — Capture pipeline

`run_with_cache` returns the same shape as `boot_transformers`: per-hook tensors keyed by canonical name, all `(1, seq, d_model)`.

In [ ]:
tokens = bridge.to_tokens(PROMPT)
logits, cache = bridge.run_with_cache(tokens)

print(f"Logits: shape={tuple(logits.shape)}, dtype={logits.dtype}")
print(f"Cache hooks: {len(cache)}")
for hk in ["blocks.0.hook_out", "blocks.0.attn.hook_out", "blocks.0.mlp.hook_out"]:
    print(f"  {hk}: shape={tuple(cache[hk].shape)} dtype={cache[hk].dtype}")

## Step 4 — Greedy parity (acceptance gate)

Next-token argmax must match `boot_transformers` exactly. SGLang's sampler bypasses `lm_head.__call__` so the driver synthesizes the logits from the sampler's top-k log-probs — argmax-correct, absolute scale off.

In [ ]:
argmax_sglang = int(logits[0, -1].argmax())
parity = argmax_sglang == argmax_hf
print(f"SGLang next-token argmax: {argmax_sglang} → {bridge.tokenizer.decode([argmax_sglang])!r}")
print(f"HF next-token argmax:     {argmax_hf}")
print(f"Greedy parity: {'✓' if parity else '✗'}")
assert parity, "Acceptance gate 2 failed: argmax mismatch vs boot_transformers"

## Step 5 — Per-hook L2 (acceptance gate)

Relative L2 distance per fireable hook must stay under `5e-3` vs HF activations. Larger drifts indicate the capture is going through the wrong path — usually a fused-kernel ordering issue in SGLang's overlay.

In [ ]:
TARGET_REL_L2 = 5e-3
worst_hook, worst_l2 = None, 0.0
for hk in sorted(bridge._driver.supported_hook_points):
    if hk not in cache or hk not in hf_acts:
        continue
    sglang_t = cache[hk].float().cpu()
    hf_t = hf_acts[hk]
    rel_l2 = ((sglang_t - hf_t).norm() / (hf_t.norm() + 1e-9)).item()
    flag = "" if rel_l2 < TARGET_REL_L2 else " ← over"
    print(f"  {hk:40s} rel_L2 = {rel_l2:.2e}{flag}")
    if rel_l2 > worst_l2:
        worst_l2, worst_hook = rel_l2, hk
print(f"\nWorst: {worst_hook} = {worst_l2:.2e} (target < {TARGET_REL_L2:.0e})")
assert worst_l2 < TARGET_REL_L2, f"Acceptance gate 3 failed: {worst_hook} = {worst_l2:.2e}"

## Step 6 — Intervention smoke (load-bearing)

Each of `suppress` / `scale` / `add` / `set` must shift the next-token argmax and revert on a clean forward (interventions aren't sticky). This proves the affine path through SGLang's compiled forward.

In [ ]:
# (a) suppress: zero the embedding output. Multiplicative op (scale=0).
supp_logits = bridge.forward(tokens, intervene={"embed.hook_out": {"op": "suppress"}})
argmax_supp = int(supp_logits[0, -1].argmax())
print(f"suppress(embed.hook_out): argmax = {argmax_supp} (clean = {argmax_sglang}, shifted = {argmax_supp != argmax_sglang})")

# (b) scale: double the residual stream at layer 0. Multiplicative op (scale=2).
scale_logits = bridge.forward(tokens, intervene={"blocks.0.hook_out": {"op": "scale", "factor": 2.0}})
argmax_scale = int(scale_logits[0, -1].argmax())
print(f"scale(blocks.0.hook_out, 2.0): argmax = {argmax_scale} (shifted = {argmax_scale != argmax_sglang})")

# (c) add: shift all residual dims by a constant.
add_logits = bridge.forward(tokens, intervene={"blocks.0.hook_out": {"op": "add", "value": 0.5}})
argmax_add = int(add_logits[0, -1].argmax())
print(f"add(blocks.0.hook_out, 0.5): argmax = {argmax_add} (shifted = {argmax_add != argmax_sglang})")

# (d) set: replace residual with a constant vector (extreme — drives the model off-manifold).
set_logits = bridge.forward(tokens, intervene={"blocks.0.hook_out": {"op": "set", "value": 0.0}})
argmax_set = int(set_logits[0, -1].argmax())
print(f"set(blocks.0.hook_out, 0): argmax = {argmax_set} (shifted = {argmax_set != argmax_sglang})")

# Clean forward — interventions must NOT persist.
clean_logits = bridge.forward(tokens)
argmax_clean = int(clean_logits[0, -1].argmax())
print(f"\nclean revert: argmax = {argmax_clean} (matches Step 4 = {argmax_clean == argmax_sglang})")
assert argmax_clean == argmax_sglang, "Acceptance gate 4 failed: intervention leaked into next forward"

## Step 7 — Hook fires exactly once

Piecewise CUDA Graph capture can double-replay hook code if the FX trace records it twice. The worker counter increments on each fire; one fire per hook per forward is correct.

In [ ]:
from transformer_lens.model_bridge.sources.sglang import rpc as sglang_rpc

n_hooks = len(bridge._driver.supported_hook_points)
sglang_rpc.reset_counter(bridge._driver._engine)
_ = bridge.forward(tokens)
count = sglang_rpc.read_counter(bridge._driver._engine)
print(f"hook fires after one forward: {count} (expected {n_hooks})")
assert count == n_hooks, f"Acceptance gate 5 failed: {count} fires for {n_hooks} hooks (double-fire?)"
print("✓ Each hook fires exactly once.")

## Step 8 — Lifetime

`bridge.close()` must release GPU memory and tear down SGLang's scheduler subprocesses.

In [ ]:
before = torch.cuda.memory_allocated() / 1e9
bridge.close()
gc.collect()
torch.cuda.empty_cache()
after = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory: before close = {before:.2f} GB, after close = {after:.2f} GB")
print(f"Released: {(before - after):.2f} GB")

## Summary

If all five acceptance gates passed, the SGLang driver is correctly extracting activations and applying interventions through SGLang's compiled forward path, with semantic parity to `boot_transformers` within fp16 tolerance.

Next: `Inspect_SGLang_Provider_Demo.ipynb` exercises the `tl_bridge_sglang` Inspect provider — the same engine as boot_sglang, surfaced as an `inspect_ai` ModelAPI for dataset-scale evals.